# Feature Engineering

In [1]:
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

In [2]:
# Sessao Spark
spark = SparkSession.builder.appName("ifood-case").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/mnt/HD2/Data_Science/ifood-case/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/26 12:17:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# Leitura do dataset processado
offer_instances = spark.read.parquet("../data/processed/offer_instances_final.parquet")

In [ ]:
# Transacoes brutas, pois vai ser necessario para o historico de compras
transactions_raw = spark.read.option("multiline", "true").json(
    "../data/raw/transactions.json"
)

transaction_events = transactions_raw.filter(F.col("event") == "transaction").select(
    "account_id",
    F.col("time_since_test_start").alias("transaction_time"),
    "value.amount",
)

In [7]:
print(f"offer_instances: {offer_instances.count()} linhas")
print(f"transaction_events: {transaction_events.count()} linhas")

offer_instances: 63288 linhas
transaction_events: 138953 linhas


In [9]:
offer_instances.show(5, truncate=False)

+--------------------------------+--------------------------------+-------------+----------------------------+--------------+---------+--------+-------------+-----------+--------------+----------+------+-------------------------+-------+----+------+-----------------+-------------+
|account_id                      |offer_id                        |offer_type   |channels                    |discount_value|min_value|duration|received_time|viewed_time|completed_time|window_end|reward|has_post_view_transaction|success|age |gender|credit_card_limit|registered_on|
+--------------------------------+--------------------------------+-------------+----------------------------+--------------+---------+--------+-------------+-----------+--------------+----------+------+-------------------------+-------+----+------+-----------------+-------------+
|0009655768c64bdeb2e877511632db8f|2906b810c7d4411798c6938adc9daaa5|discount     |[web, email, mobile]        |2             |10       |7.0     |24.0      

In [8]:
transaction_events.show(5, truncate=False)

+--------------------------------+----------------+------+
|account_id                      |transaction_time|amount|
+--------------------------------+----------------+------+
|02c083884c7d45b39cc68e1314fec56c|0.0             |0.83  |
|9fa9ae8f57894cc9a3b8a9bbe0fc1b2f|0.0             |34.56 |
|54890f68699049c2a04d415abc25e717|0.0             |13.23 |
|b2f1cd155b864803ad8334cdf13c4bd2|0.0             |19.51 |
|fe97aa22dd3e48c8b143116a8403dd52|0.0             |18.97 |
+--------------------------------+----------------+------+
only showing top 5 rows


In [10]:
# Confirmacao do final do periodo de teste
test_end = transactions_raw.select(F.max("time_since_test_start")).first()[0]
print(f"Final do periodo de teste: {test_end}")

Final do periodo de teste: 29.75


In [ ]:
# Aqui, a ideia vai ser criar 3 features novas.
# is_profile_completed: essa feature vai indicar se o cliente possui cadastro completo ou nao
# registration_year: essa feature vai indicar a tenure do cliente
# remaining_days: essa feature vai indicar quanto sobrou do periodo de teste apos o fim da janela da oferta

offer_features = (
    offer_instances.withColumn(
        "is_profile_completed", F.col("age").isNotNull().cast("int")
    )
    .withColumn("registration_year", F.year("registered_on"))
    .withColumn("remaining_days", F.lit(test_end) - F.col("window_end"))
)

In [12]:
# Verificacao do resultado
offer_features.groupBy("is_profile_completed").count().show()

+--------------------+-----+
|is_profile_completed|count|
+--------------------+-----+
|                   1|55222|
|                   0| 8066|
+--------------------+-----+



In [14]:
offer_features.select("registration_year", "remaining_days").summary().show()

+-------+------------------+------------------+
|summary| registration_year|    remaining_days|
+-------+------------------+------------------+
|  count|             63288|             63288|
|   mean|2016.6376090254078|10.471511186954872|
| stddev|1.1785579051880408| 8.584410934470272|
|    min|              2013|             -4.25|
|    25%|              2016|              2.75|
|    50%|              2017|              8.75|
|    75%|              2017|             17.75|
|    max|              2018|             26.75|
+-------+------------------+------------------+



In [ ]:
# Agora, vai ser feito o encoding de "channels". A ideia e criar uma feature booleana para cada canal
# Um detalhe: "email" vai ser removido por existir em todas as ofertas
offer_features = (
    offer_features.withColumn(
        "channel_mobile", F.array_contains("channels", "mobile").cast("int")
    )
    .withColumn("channel_social", F.array_contains("channels", "social").cast("int"))
    .withColumn("channel_web", F.array_contains("channels", "web").cast("int"))
    .drop("channels")
)

In [ ]:
# Validacao - bate com a validacao feita na EDA 05
offer_features.select(
    F.sum("channel_mobile").alias("mobile"),
    F.sum("channel_social").alias("social"),
    F.sum("channel_web").alias("web"),
    F.count("*").alias("total"),
).show()

+------+------+-----+-----+
|mobile|social|  web|total|
+------+------+-----+-----+
| 56914| 37943|50594|63288|
+------+------+-----+-----+



In [18]:
# A ideia vai ser olhar para a janela por "account_id"
# Para cada instancia, olhar as ofertas anteriores do mesmo cliente
customer_history_window = (
    Window.partitionBy("account_id")
    .orderBy("received_time")
    .rowsBetween(Window.unboundedPreceding, -1)
)

In [ ]:
# Agora, a partir disso, a ideia vai ser criar features do historico
# offers_received_prior = numero de ofertas recebidas pelo cliente antes da oferta atual
# prior_success_rate = taxa de sucesso do cliente em ofertas anteriores
# prior_view_rate = taxa de visualizacao do cliente em ofertas anteriores
offer_features = (
    offer_features.withColumn(
        "offers_received_prior", F.count("*").over(customer_history_window)
    )
    .withColumn("prior_success_rate", F.avg("success").over(customer_history_window))
    .withColumn(
        "prior_view_rate",
        F.avg(F.col("viewed_time").isNotNull().cast("int")).over(
            customer_history_window
        ),
    )
)

In [ ]:
# Validacao
# Escolhe um cliente com mais de uma oferta para verificar a logica de olhar para o passado
sample_account = (
    offer_features.groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .first()["account_id"]
)

offer_features.filter(F.col("account_id") == sample_account).select(
    "account_id",
    "received_time",
    "success",
    "viewed_time",
    "offers_received_prior",
    "prior_success_rate",
    "prior_view_rate",
).orderBy("received_time").show(truncate=False)

+--------------------------------+-------------+-------+-----------+---------------------+------------------+---------------+
|account_id                      |received_time|success|viewed_time|offers_received_prior|prior_success_rate|prior_view_rate|
+--------------------------------+-------------+-------+-----------+---------------------+------------------+---------------+
|01e09d713abe4a36a70a33fe4b40534e|0.0          |0      |NULL       |0                    |NULL              |NULL           |
|01e09d713abe4a36a70a33fe4b40534e|7.0          |0      |NULL       |1                    |0.0               |0.0            |
|01e09d713abe4a36a70a33fe4b40534e|14.0         |0      |NULL       |2                    |0.0               |0.0            |
|01e09d713abe4a36a70a33fe4b40534e|17.0         |1      |17.5       |3                    |0.0               |0.0            |
+--------------------------------+-------------+-------+-----------+---------------------+------------------+---------

In [ ]:
# Aqui, a ideia vai ser gerar um formato comum para os datasets

# Criar o marcador do evento de transacao, que vai ser tempo + valor da compra
transaction_marker = transaction_events.select(
    "account_id",
    F.col("transaction_time").alias("event_time"),
    F.lit(1).alias("is_transaction"),
    "amount",
)

# Criar o marcador do evento de oferta recebida, que vai usar offer_id + received_time
offer_marker = offer_features.select(
    "account_id",
    "offer_id",
    F.col("received_time").alias("event_time"),
    F.lit(0).alias("is_transaction"),
    F.lit(None).cast("double").alias("amount"),
)

In [ ]:
# Agora, a ideia vai ser unir os dois markers numa linha temporal por cliente
transaction_marker = transaction_marker.withColumn(
    "offer_id", F.lit(None).cast("string")
)

# Aqui sim acontece o union das duas timelines por account_id
account_timeline = transaction_marker.unionByName(offer_marker)

In [24]:
# Validacao
print(f"account_timeline: {account_timeline.count()} linhas")
print(f"esperado: {transaction_marker.count() + offer_marker.count()} linhas")

account_timeline.orderBy("account_id", "event_time").show(10, truncate=False)

account_timeline: 202241 linhas
esperado: 202241 linhas


+--------------------------------+----------+--------------+------+--------------------------------+
|account_id                      |event_time|is_transaction|amount|offer_id                        |
+--------------------------------+----------+--------------+------+--------------------------------+
|0009655768c64bdeb2e877511632db8f|7.0       |0             |NULL  |5a8bc65990b245e5a138643cd4eb9837|
|0009655768c64bdeb2e877511632db8f|9.5       |1             |22.16 |NULL                            |
|0009655768c64bdeb2e877511632db8f|14.0      |0             |NULL  |3f207df678b143eea3cee63160fa8bed|
|0009655768c64bdeb2e877511632db8f|17.0      |0             |NULL  |f19421c1d4aa40978ebb69ca19b0e20d|
|0009655768c64bdeb2e877511632db8f|17.25     |1             |8.57  |NULL                            |
|0009655768c64bdeb2e877511632db8f|21.0      |0             |NULL  |fafdcd668e3743c1bb461111dcafc2a4|
|0009655768c64bdeb2e877511632db8f|22.0      |1             |14.11 |NULL                    

In [25]:
# Agora, a ideia vai ser criar uma janela agregada
account_history_window = (
    Window.partitionBy("account_id")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, -1)
)

In [ ]:
# Nessa agragacao, so contam quando is_transaction == 1, olhando somente para o historico de compras passadas do cliente
account_timeline = (
    account_timeline.withColumn(
        "txn_count_prior", F.sum("is_transaction").over(account_history_window)
    )
    .withColumn(
        "txn_amount_sum_prior",
        F.sum(F.when(F.col("is_transaction") == 1, F.col("amount"))).over(
            account_history_window
        ),
    )
    .withColumn(
        "txn_amount_avg_prior", F.col("txn_amount_sum_prior") / F.col("txn_count_prior")
    )
    .withColumn(
        "last_txn_time_prior",
        F.max(F.when(F.col("is_transaction") == 1, F.col("event_time"))).over(
            account_history_window
        ),
    )
    .withColumn(
        "days_since_last_txn", F.col("event_time") - F.col("last_txn_time_prior")
    )
)

In [27]:
# Validacao
account_timeline.filter(
    F.col("account_id") == "0009655768c64bdeb2e877511632db8f"
).select(
    "event_time",
    "is_transaction",
    "amount",
    "txn_count_prior",
    "txn_amount_sum_prior",
    "txn_amount_avg_prior",
    "days_since_last_txn",
).orderBy("event_time").show(truncate=False)

+----------+--------------+------+---------------+--------------------+--------------------+-------------------+
|event_time|is_transaction|amount|txn_count_prior|txn_amount_sum_prior|txn_amount_avg_prior|days_since_last_txn|
+----------+--------------+------+---------------+--------------------+--------------------+-------------------+
|7.0       |0             |NULL  |NULL           |NULL                |NULL                |NULL               |
|9.5       |1             |22.16 |0              |NULL                |NULL                |NULL               |
|14.0      |0             |NULL  |1              |22.16               |22.16               |4.5                |
|17.0      |0             |NULL  |1              |22.16               |22.16               |7.5                |
|17.25     |1             |8.57  |1              |22.16               |22.16               |7.75               |
|21.0      |0             |NULL  |2              |30.73               |15.365              |3.75

In [28]:
# Extracao das features de "is_transaction"=0.
# Com isso, mantem-se as instancias das ofertas.
transaction_history = account_timeline.filter(F.col("is_transaction") == 0).select(
    "account_id",
    "offer_id",
    "txn_count_prior",
    "txn_amount_sum_prior",
    "txn_amount_avg_prior",
    "days_since_last_txn",
)

In [ ]:
# Join por (account_id, offer_id)
offer_features = offer_features.join(
    transaction_history, on=["account_id", "offer_id"], how="left"
)

In [30]:
# Validacao
print(f"offer_features: {offer_features.count()} linhas")

offer_features.select(
    "txn_count_prior",
    "txn_amount_sum_prior",
    "txn_amount_avg_prior",
    "days_since_last_txn",
).summary().show()

offer_features: 63288 linhas


26/07/26 15:01:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+--------------------+--------------------+-------------------+
|summary|   txn_count_prior|txn_amount_sum_prior|txn_amount_avg_prior|days_since_last_txn|
+-------+------------------+--------------------+--------------------+-------------------+
|  count|             49485|               44194|               44194|              44194|
|   mean| 4.084853996160453|  58.019311218717554|  13.414852510617319| 3.0563651174367563|
| stddev|3.5359016957030502|   84.12677659076378|  19.635054563642168| 3.0940347505036416|
|    min|                 0|                0.05|                0.05|                0.0|
|    25%|                 1|               11.97|              3.1775|               0.75|
|    50%|                 3|               32.06|               11.35|               2.25|
|    75%|                 6|   78.24000000000001|              19.894|                4.5|
|    max|                29|  1325.7200000000003|              855.31|               24.0|

In [31]:
# Selecao das features finais para o modelo
model_features = offer_features.select(
    "account_id",
    "offer_id",
    "offer_type",
    "discount_value",
    "min_value",
    "duration",
    "channel_mobile",
    "channel_social",
    "channel_web",
    "received_time",
    "remaining_days",
    "age",
    "gender",
    "credit_card_limit",
    "is_profile_completed",
    "registration_year",
    "offers_received_prior",
    "prior_success_rate",
    "prior_view_rate",
    "txn_count_prior",
    "txn_amount_sum_prior",
    "txn_amount_avg_prior",
    "days_since_last_txn",
    "success",
)

In [32]:
model_features.show(5, truncate=False)

+--------------------------------+--------------------------------+-------------+--------------+---------+--------+--------------+--------------+-----------+-------------+--------------+----+------+-----------------+--------------------+-----------------+---------------------+------------------+---------------+---------------+--------------------+--------------------+-------------------+-------+
|account_id                      |offer_id                        |offer_type   |discount_value|min_value|duration|channel_mobile|channel_social|channel_web|received_time|remaining_days|age |gender|credit_card_limit|is_profile_completed|registration_year|offers_received_prior|prior_success_rate|prior_view_rate|txn_count_prior|txn_amount_sum_prior|txn_amount_avg_prior|days_since_last_txn|success|
+--------------------------------+--------------------------------+-------------+--------------+---------+--------+--------------+--------------+-----------+-------------+--------------+----+------+----

In [33]:
model_features.write.mode("overwrite").parquet(
    "../data/processed/model_features.parquet"
)

In [ ]:
model_features_check = spark.read.parquet("../data/processed/model_features.parquet")

print(
    f"linhas: {model_features_check.count()}; colunas: {len(model_features_check.columns)}"
)

model_features_check.printSchema()

linhas: 63288; colunas: 24
root
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- offer_type: string (nullable = true)
 |-- discount_value: long (nullable = true)
 |-- min_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- channel_mobile: integer (nullable = true)
 |-- channel_social: integer (nullable = true)
 |-- channel_web: integer (nullable = true)
 |-- received_time: double (nullable = true)
 |-- remaining_days: double (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- is_profile_completed: integer (nullable = true)
 |-- registration_year: integer (nullable = true)
 |-- offers_received_prior: long (nullable = true)
 |-- prior_success_rate: double (nullable = true)
 |-- prior_view_rate: double (nullable = true)
 |-- txn_count_prior: long (nullable = true)
 |-- txn_amount_sum_prior: double (nullable = true)
 |-- txn_amount_avg_prio